In [4]:
import pydantic
from pydantic import validate_call

In [5]:
@validate_call
def add_numb(a:int, b:int):
    return a+b

In [ ]:
add_numb('2',3) # pydantic convert str into int

5

In [ ]:
from pydantic import StrictInt

@validate_call
def add_numb(a:StrictInt, b:StrictInt):
    return a+b


add_numb('12',3) #-> this will give error

'2A13'

In [ ]:
from pydantic import PositiveInt

# only postive number greater than zero 

# zero it will give error



In [10]:
from pydantic import BaseModel

class User(BaseModel):
    user_id:int
    name:str
    age:int
    is_active:bool


user1=User(user_id=1,name="Krushna",age=34,is_active=True)
user1


User(user_id=1, name='Krushna', age=34, is_active=True)

In [12]:
user2=User(user_id=1,name="Krushna",age=34,is_active='aka')
user2

ValidationError: 1 validation error for User
is_active
  Input should be a valid boolean, unable to interpret input [type=bool_parsing, input_value='aka', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/bool_parsing

In [16]:
from pydantic import PositiveInt,BaseModel
from typing import List, Literal, Optional
class Orders(BaseModel):
    user_id:PositiveInt
    product_Ids:List[int]
    payment_mode:Literal["Online","Cash","UPI"]
    delivery_note:Optional[str]=None


order1=Orders(user_id=3,product_Ids=[3,65,7],payment_mode='UPI')
order1

Orders(user_id=3, product_Ids=[3, 65, 7], payment_mode='UPI', delivery_note=None)

#FIELD VALIDATION


In [ ]:
# suppose name its max and min length limit

from pydantic import BaseModel, Field

class Employee(BaseModel):
    name:str= Field(
        ...,
        min_length=6,
        max_length=20,
        description="Employee Name"
    )
    salary:int= Field(
        ..., ge=25000,le=250000
    )
    email:str=Field(default="") # actiing like optinal

emp1=Employee(name='adfgdsd',salary=167532)
emp1

Employee(name='adfgdsd', salary=167532, email='')

# Field Validator & Model Validator

In [25]:
from pydantic import BaseModel, Field, field_validator, model_validator

# model validator can validate all the parameters

class Register(BaseModel):
    username:str=Field(..., min_length=8)
    passward:str
    re_passward:str
    mobile:int

   
    @field_validator('mobile')
    def validate_mobile(cls,value):
        if len(str(value))!=10:
            return ValueError("Mobile number must be 10 digit")
        
        return value
    
    @model_validator(mode='after')  # after defining the valeus
    def pass_validation(self):
        if self.passward!=self.re_passward:
            raise ValueError("Passward do not match")
        
        if len(self.passward)<8:
            raise ValueError('Password min length should be 8')

        return self
    

new_reg=Register(username='kborudehuhh',passward='12345678',re_passward='12345679',mobile=60875)
new_reg

ValidationError: 1 validation error for Register
  Value error, Passward do not match [type=value_error, input_value={'username': 'kborudehuhh...45679', 'mobile': 60875}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [26]:
class Even(BaseModel):
    name:str
    start_date:int
    end_data:int

    @model_validator(mode="after")
    def valid_dates(self):
        if self.start_date>self.end_data:
            raise ValueError('Start date must be before end date')
        
        return self


event1=Even(name='ai course',start_date=25, end_data=23)


ValidationError: 1 validation error for Even
  Value error, Start date must be before end date [type=value_error, input_value={'name': 'ai course', 'st...te': 25, 'end_data': 23}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

##Computed Power

In [28]:
from pydantic import BaseModel, computed_field


class CartItem(BaseModel):
    user_id:int
    product_id:int
    price:int
    quantity:int

    @computed_field
    @property
    def total_amound(self)-> int:
        return self.price*self.quantity
    

cart=CartItem(user_id=1,product_id=1,price=120,quantity=4)

print(cart)


# write a function which calcuating total cart value

user_id=1 product_id=1 price=120 quantity=4 total_amound=480


##ANNOTETED

In [32]:
from typing import Annotated


class User(BaseModel):
    # name:str=Field(..., min_length=8)  you can update this using anootated
    # age:int=Field(..., ge=18)
    name:Annotated[str,Field(..., min_length=8)]
    age:Annotated[int,Field(..., ge=18)]



user=User(name="Hellow_gunda hu",age=19)
user

User(name='Hellow_gunda hu', age=19)

## SERIALIZATION & DESERALIATION

In [33]:

class Students(BaseModel):
    name:str
    email:str
    age:int
    branch:str
    sem:int
    address:str

data={
    "name":'krushna',
    "email":"kersuhn@gmail.com",
    "age":23,
    "branch":"CSE",
    "sem":4,
    "address":"Patna"
}


# student=Students(**data)   # ** basically de strcuturing the args qrgs
student1=Students.model_validate(data)

student1


Students(name='krushna', email='kersuhn@gmail.com', age=23, branch='CSE', sem=4, address='Patna')

In [34]:
student1.model_dump()

{'name': 'krushna',
 'email': 'kersuhn@gmail.com',
 'age': 23,
 'branch': 'CSE',
 'sem': 4,
 'address': 'Patna'}

In [ ]:
# serialization when convert into dic and deserialzation when converted
# from dic to pydentic obj

## SETTINGS MANAGMENT

In [ ]:
# if we work on big project

# we don't want to show API keys to other

from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env")

    API_KEY:str
    DB_USERNAME:str 
    DB_PASSWORD:str

settings = Settings()
print(settings.API_KEY)